[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amark-23/fno-pde/blob/main/notebooks/navier_stokes_rollout_colab.ipynb)

# FNO Part 2b: rollout-stabilized training

Compares three training schemes for the 2D Navier-Stokes FNO on their
autoregressive rollout error: the single-step **baseline**, **pushforward**
(unroll several steps, gradient on the last), and **noise injection**. Each is a
config in `configs/`; the training logic is in `src/train_ns.py`.

The goal is to flatten the rollout error-growth curve from Concept 17. Run top to
bottom on a T4 GPU.

> Note: this clones the repo from GitHub, so make sure `src/train_ns.py` and the
> `configs/ns_*.yaml` files are pushed before running.

## 1. Clone and install

In [ ]:
!git clone https://github.com/amark-23/fno-pde.git
%cd fno-pde
!pip -q install pyyaml
import sys; sys.path.insert(0, '/content/fno-pde')
import torch; print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Dataset from Drive
Uses the same `navier_stokes.npz` generated for the training notebook.

In [ ]:
import os
from google.colab import drive; drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/navier_stokes.npz'
assert os.path.exists(DATA), 'navier_stokes.npz not found on Drive'
print('dataset:', DATA)

## 3. Train the three variants
Each loads its config, points it at the Drive dataset, and trains. Reduce
`epochs` in the configs if a full run is too slow.

In [ ]:
import yaml
from src.train_ns import train_rollout

device = 'cuda' if torch.cuda.is_available() else 'cpu'
configs = {
    'baseline':    'configs/ns_baseline.yaml',
    'pushforward': 'configs/ns_pushforward.yaml',
    'noise':       'configs/ns_noise.yaml',
}
results, models = {}, {}
for name, path in configs.items():
    cfg = yaml.safe_load(open(path))
    cfg['data'] = DATA                       # point the config at the Drive dataset
    print(f'=== {name} ===')
    model, res = train_rollout(cfg, device)
    results[name], models[name] = res, model


## 4. Compare rollout error growth
The headline figure: relative L2 versus rollout step for each scheme.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6.2, 4.0))
for name, res in results.items():
    c = res['rollout_curve']
    plt.plot(range(1, len(c) + 1), c, '-o', ms=4, label=name)
plt.xlabel('rollout step'); plt.ylabel('relative L2')
plt.title('Rollout error growth by training scheme')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/content/rollout_stability.png', dpi=130)
from google.colab import files; files.download('/content/rollout_stability.png')
plt.show()

for name, res in results.items():
    c = res['rollout_curve']
    print(f"{name:12s} step1 {c[0]:.3f}  end {c[-1]:.3f}")

## 5. Save the trained models (optional)
Saves the three model state dicts to Drive so the comparison can be reloaded
without retraining.

In [ ]:
for name, model in models.items():
    path = f'/content/drive/MyDrive/ns_{name}.pt'
    torch.save(model.state_dict(), path)
    print('saved', path)
